# SCD TYPE 2

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from delta.tables import *

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pyspark_cata_vijay.source.customers(
    customer_id string,
    email string,
    name string,
    city string,
    country string,
    delivery_updates string,
    order_id string,
    modified_date string,
    amount string,
    currency string
);


INSERT INTO pyspark_cata_vijay.source.customers
values
('1','abc@yahoomail.com','Jane','New York','USA','true','1',current_timestamp(),'100','USD'),
('2','def@yahoomail.com','John','New York','USA','true','1',current_timestamp(),'200','USD'),
('3','ghi@yahoomail.com','Mary','New York','USA','true','1',current_timestamp(),'300','USD'),
('4','jkl@yahoomail.com','Peter','New York','USA','true','1',current_timestamp(),'400','USD'),
('5','mno@yahoomail.com','Sara','New York','USA','true','1',current_timestamp(),'500','USD'),
('6','pqr@yahoomail.com','Tom','New York','USA','true','1',current_timestamp(),'600','USD'),
('7','stu@yahoomail.com','Alice','New York','USA','true','1',current_timestamp(),'700','USD'),
('8','vwx@yahoomail.com','Bob','New York','USA','true','1',current_timestamp(),'800','USD'),
('9','yz@yahoomail.com','Charlie','New York','USA','true','1',current_timestamp(),'900','USD'),
('10','abc@yahoomail.com','Jane','New York','USA','true','1',current_timestamp(),'1000','USD')


In [0]:
%sql
Select * from  pyspark_cata_vijay.source.customers;

In [0]:
if spark.catalog.tableExists("pyspark_cata_vijay.source.Dimcustomers"):
    pass
else:
    spark.sql("""
              CREATE TABLE IF NOT EXISTS pyspark_cata_vijay.source.Dimcustomers
              SELECT *,
              current_timestamp() as Start_date,
              cast('9999-12-31' AS TIMESTAMP) as End_Date,
              'Y' AS IsActive
              from pyspark_cata_vijay.source.customers
              """
              )
    

In [0]:
%sql
SELECT * FROM pyspark_cata_vijay.source.dimcustomers


# **Merge 1 - Marking updated records as EXPIRED ( Records which got updated by new records)**

In [0]:
df = spark.sql("""
               SELECT * from pyspark_cata_vijay.source.customers
               """)
df =df.withColumn('dedup', row_number().over(Window.partitionBy('customer_id').orderBy(col('modified_date').desc())))
df = df.filter(df.dedup == 1).drop('dedup')

df.createOrReplaceTempView("src")

In [0]:
%sql
MERGE INTO pyspark_cata_vijay.source.dimcustomers as trg
USING src
ON src.customer_id = trg.customer_id
AND trg.isActive = 'Y'
WHEN MATCHED 
AND src.email <> trg.email
OR src.name <> trg.name
OR src.city <> trg.city
OR src.country <> trg.country
OR src.modified_date <> trg.modified_date
THEN UPDATE SET 
trg.end_date = current_timestamp(),
trg.isactive = 'N'

# Merge 2- Inserting New Records as well as the updated records

In [0]:
%sql
--Inserting new records as well as Updated Records
MERGE INTO pyspark_cata_vijay.source.dimcustomers as trg
USING src
ON src.customer_id = trg.customer_id
AND trg.isActive = 'Y'
WHEN NOT MATCHED THEN INSERT *
;